# ARENA3 정책 반복 1라운드 — 학생 학습·심판 평가·판정

`ARENA3_PI_R1_data`의 64판이 모두 끝난 뒤 **런타임 하나**(CPU + 높은 RAM)에서 **모두 실행**합니다.
1. 날짜 보류 진단(참고, 약 10분): 학습에 안 쓴 날짜에서 '더 나은 녹색'을 얼마나 맞히는지 봅니다. 판정·선택에는 쓰지 않습니다.
2. 학생 학습(약 5분): 같은 자료로 난수 42·43·44 세 신경망을 학습합니다. 입력은 관측 119칸뿐입니다.
3. 심판 평가: 세 변형 × 모델 3개 × 84상황 = 756판(CPU 8개 기준 약 5~6시간). 1단계와 같은 심판·같은 84상황입니다.
   - **R1M**: 갈림 지점에서 모드가 내는 녹색만(지금 행동 범위 안의 정책 반복)
   - **R1G**: 갈림 지점에서 허용 녹색 전부(새 행동)
   - **R1A**: 갈림·합의 지점 모두에서 허용 녹색 전부
4. 판정: M4 대비 판정 + 짝 비교(R1M − S2A_pre, R1G − R1M, R1A − R1G, 각 변형 − S2B, 각 변형 − B′).
5. 확인: 통과한 변형 중 날짜 평균이 가장 큰 하나만 C1·C2 84상황에서 다시 잽니다(약 2시간). 통과가 없으면 SUMO를 돌리지 않고 끝납니다.

빨리 끝내려면: 2번(학생 학습)이 끝난 뒤 다른 런타임 두 곳에서 이 노트북을 열고 첫 칸의 `VARIANTS`를 각각 `("G",)`, `("A",)`로,
이 런타임은 `("M",)`으로 바꿔 3번 칸까지 실행하세요. 4번(판정)·5번(확인)은 세 평가가 끝난 뒤 한 곳에서만 실행합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_PI.zip"
A3 = DRIVE / "arena3"
OUT = A3 / "pi" / "R1"
WORKSPACE = Path("/content/FutureTraffic_ARENA3_PI")
MIN_CPUS = 8
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1

VARIANTS = ("M", "G", "A")      # 이 런타임에서 평가할 변형(나눠 돌릴 때만 바꿉니다)
print("CPU 개수:", CPUS, "| 결과 폴더:", OUT, "| 평가할 변형:", VARIANTS)
if CPUS < MIN_CPUS:
    print(f"주의: CPU가 {CPUS}개입니다. 평가가 오래 걸립니다. 가능하면 CPU + 높은 RAM 런타임을 쓰세요.")
done = len(list((OUT / "data" / "runs" / "PI_DATA").glob("*/cache.json")))
assert done == 64, f"자료가 64판 중 {done}판만 끝났습니다. ARENA3_PI_R1_data를 먼저 끝내세요."

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_PI_R1", "ARENA3 정책 반복 1라운드 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| 정책 반복 설정 {package_manifest['pi_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 날짜 보류 진단(참고, SUMO 없음).
OFFLINE = OUT / "offline"
if not (OFFLINE / "offline_report.md").exists():
    colab.run_live(["src/arena3_pi.py", "offline", "--data", str(OUT / "data"), "--out", str(OFFLINE)], WORKSPACE)
display(Markdown((OFFLINE / "offline_report.md").read_text()))

In [ ]:
# 2) 학생 학습(난수 42·43·44, SUMO 없음). 이미 있으면 같은 자료로 학습했는지만 확인하고 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
colab.run_live(["src/arena3_pi.py", "train", "--data", str(OUT / "data"), "--out", str(OUT / "students")], WORKSPACE)

In [ ]:
# 3) 심판 평가: 변형마다 모델 3개 × 84상황 = 252판. 끊겨도 다시 실행하면 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
missing = [s for s in (42, 43, 44) if not (OUT / "students" / f"seed_{s}" / "model.pt").exists()]
assert not missing, f"학생 모델이 없습니다(난수 {missing}). 2)를 먼저 실행하세요."
for v in VARIANTS:
    colab.run_live(["src/arena3_pi.py", "evaluate", "--variant", v, "--students", str(OUT / "students"),
                    "--out", str(OUT / "eval" / f"R1{v}"), "--work", f"/content/pi_eval_{v}", "--workers", str(CPUS)],
                   WORKSPACE)
    print(f"R1{v} 평가 끝")

In [ ]:
# 4) 판정(SUMO 없음, 몇 분). 세 변형의 평가가 모두 끝나야 합니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
S2 = A3 / "stage2"
args = ["src/arena3_pi.py", "judge", "--baseline", str(A3 / "referee" / "baseline"), "--out", str(OUT / "judge"),
        "--reference", f"BPRIME={A3 / 'referee' / 'bprime'}"]
for v in ("M", "G", "A"):
    args += ["--pi", f"R1{v}={OUT / 'eval' / f'R1{v}'}"]
for n in ("S2A_pre", "S2A", "S2B_pre", "S2B"):
    args += ["--reference", f"{n}={S2 / 'eval' / n}"]
code = colab.run_live(args, WORKSPACE, check=False)
assert code == 0, "아직 평가가 끝나지 않은 변형이 있습니다(위 '아직 평가 중'). 평가가 끝난 뒤 다시 실행하세요."
display(Markdown((OUT / "judge" / "summary.md").read_text()))

In [ ]:
# 5) 확인(C1·C2 84상황): 통과한 변형 중 날짜 평균이 가장 큰 하나만. 승자는 처음 한 번 고정되어, 다시 실행해도 바뀌지 않습니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요."
CONFIRM_BASE = A3 / "stage2" / "confirm" / "baseline"     # 2단계 확인 때 돌린 M4·P0·W4(C1·C2) 252판
colab.run_live(["src/arena3_pi.py", "confirm", "--judge", str(OUT / "judge"), "--baseline", str(CONFIRM_BASE),
                "--reference", f"S2B={A3 / 'stage2' / 'confirm' / 'S2B'}", "--out", str(OUT / "judge" / "confirm"),
                "--work", "/content/pi_confirm", "--workers", str(CPUS)], WORKSPACE)
result = colab.read_json(OUT / "judge" / "confirm" / "confirmation.json")
print("확인 결과:", result["status"], "| 승자:", (result.get("selection") or {}).get("winner"))
if (OUT / "judge" / "confirm" / "summary.md").exists():
    display(Markdown((OUT / "judge" / "confirm" / "summary.md").read_text()))